# Day 18: Heart Disease Prediction — EDA & Preprocessing

**Project:** Heart Disease Prediction System  

In this notebook, we will:
1. Load and explore the dataset
2. **Handle dirty data** — missing values, duplicates, irrelevant columns
3. Understand each feature's medical meaning
4. Perform Exploratory Data Analysis (EDA)
5. Visualize patterns and correlations
6. Preprocess data for ML models (scaling, train-test split)

## 1. Import Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Load the Dataset

In [4]:
df = pd.read_csv('heart_disease.csv')
print(f"Dataset Shape: {df.shape}")
print(f"Total Patients: {len(df)}")
df.head(10)

Dataset Shape: (311, 15)
Total Patients: 311


,patient_id,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,PID_0291,68.0,0,2,133.0,239.0,0,0,133.0,0,0.0,0,0,0,1
1,PID_0010,39.0,1,2,159.0,342.0,1,1,138.0,0,0.4,1,0,2,1
2,PID_0058,42.0,0,0,138.0,233.0,0,0,144.0,1,1.4,1,0,0,0
3,PID_0061,68.0,1,3,112.0,231.0,0,1,150.0,0,1.1,1,0,0,1
4,PID_0026,40.0,0,3,130.0,279.0,0,0,111.0,0,0.8,1,3,2,1
5,PID_0064,34.0,1,1,125.0,216.0,0,1,153.0,1,1.2,2,0,0,1
6,PID_0093,43.0,1,0,133.0,252.0,0,0,132.0,0,NaN,1,2,0,0
7,PID_0185,69.0,1,1,135.0,300.0,0,0,146.0,0,3.5,2,0,0,0
8,PID_0245,31.0,0,2,122.0,188.0,0,1,135.0,0,2.1,1,2,1,1
9,PID_0047,37.0,1,0,146.0,268.0,0,0,145.0,0,1.9,1,0,0,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   patient_id  311 non-null    object 
 1   age         303 non-null    float64
 2   sex         311 non-null    int64  
 3   cp          311 non-null    int64  
 4   trestbps    305 non-null    float64
 5   chol        299 non-null    float64
 6   fbs         311 non-null    int64  
 7   restecg     311 non-null    int64  
 8   thalach     301 non-null    float64
 9   exang       311 non-null    int64  
 10  oldpeak     306 non-null    float64
 11  slope       311 non-null    int64  
 12  ca          311 non-null    int64  
 13  thal        311 non-null    int64  
 14  target      311 non-null    int64  
dtypes: float64(5), int64(9), object(1)
memory usage: 36.6+ KB


In [7]:
print(f"\nMissing Values:\n{df.isnull().sum()}")


Missing Values:
patient_id     0
age            8
sex            0
cp             0
trestbps       6
chol          12
fbs            0
restecg        0
thalach       10
exang          0
oldpeak        5
slope          0
ca             0
thal           0
target         0
dtype: int64


In [9]:
print(f"\nDuplicates: {df.duplicated().sum()}")


Duplicates: 8


## 3. Data Cleaning — Handling Real-World Data Problems

Real datasets are messy! Ours has:
- **Missing values** in several columns
- **Duplicate rows** (same patient recorded twice)
- **Irrelevant columns** (patient_id — not useful for prediction)

Let's fix each one.

### 3.1 Drop Irrelevant Columns

`patient_id` is just an identifier — it has no predictive power. Keeping it would confuse the model.

In [11]:
print(f"Columns: {list(df.columns)}")
df.drop('patient_id', axis=1, inplace=True)

Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


KeyError: "['patient_id'] not found in axis"

### 3.2 Handle Duplicate Rows

Duplicates can bias the model by giving extra weight to repeated samples.

In [14]:
# Show some duplicates
print("\nSample duplicate rows:")
duplicates = df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(6)
print(duplicates.to_string())

# Remove duplicates
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"\nAfter removing duplicates: {len(df)} rows")


Sample duplicate rows:
Empty DataFrame
Columns: [age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal, target]
Index: []

After removing duplicates: 303 rows


### 3.3 Handle Missing Values

There are multiple strategies for handling missing data:

| Method | When to Use |
|--------|-------------|
| **Drop rows** | Very few missing values, large dataset |
| **Fill with Mean** | Numerical column, roughly symmetric distribution |
| **Fill with Median** | Numerical column with outliers/skewed data |
| **Fill with Mode** | Categorical columns |
| **Forward/Backward Fill** | Time-series data |


In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
for col, count in missing_cols.items():
    pct = count / len(df) * 100
    print(f"  {col:10s}: {count:2d} missing ({pct:.1f}%)")
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")
print(f"Total rows with at least one missing value: {df.isnull().any(axis=1).sum()}")

# Visualize missing values
plt.figure(figsize=(10, 4))
plt.bar(missing_cols.index, missing_cols.values, color='salmon', edgecolor='black')
plt.ylabel('Missing Count')
plt.title('Missing Values by Column')
plt.tight_layout()
plt.show()

In [ ]:
# Method 1: Fill with MEAN — good for symmetric distributions
age_mean = df['age'].mean()
print("METHOD 1: Fill with Mean")
print("-" * 40)
print(f"  Mean value: {age_mean:.1f}")
df['age'].fillna(age_mean, inplace=True)

In [ ]:
# Method 2: Fill with MEDIAN — better when outliers exist
print("METHOD 2: Fill with Median")
print("-" * 40)

for col in ['chol', 'trestbps']:
    median_val = df[col].median()
    print(f"  Column: {col}")
    print(f"  Median value: {median_val:.1f}")
    df[col].fillna(median_val, inplace=True)


In [ ]:
# Method 3: Fill with GROUP MEAN — smarter approach
# Fill thalach based on target group (disease patients have different heart rates)
print("METHOD 3: Fill with Group Mean (by target)")
print("-" * 40)

group_means = df.groupby('target')['thalach'].mean()
print(f"  Mean thalach for Healthy (0): {group_means[0]:.1f}")
print(f"  Mean thalach for Disease (1): {group_means[1]:.1f}")
print(f"  Missing before: {df['thalach'].isnull().sum()}")

df['thalach'] = df.groupby('target')['thalach'].transform(
    lambda x: x.fillna(x.mean())
)

print(f"  Missing after:  {df['thalach'].isnull().sum()} ✓")
print("\n  → Each patient gets the mean of their OWN group!")
print("  → More accurate than a single global mean.")

In [ ]:
# Method 4: Drop rows — when very few values are missing
print("METHOD 4: Drop Rows with Missing Values")
print("-" * 40)
print(f"  Column: oldpeak")
print(f"  Missing: {df['oldpeak'].isnull().sum()} rows ({df['oldpeak'].isnull().sum()/len(df)*100:.1f}%)")
print(f"  Rows before: {len(df)}")

df.dropna(subset=['oldpeak'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"  Rows after:  {len(df)}")
print("\n  → Only use this when missing % is very small (<2%)")
print("  → Otherwise you lose too much data!")

In [ ]:
# Final check — all clean!
print("FINAL DATA QUALITY CHECK:")
print("=" * 50)
print(f"  Shape: {df.shape}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Duplicates: {df.duplicated().sum()}")
print(f"  Columns: {list(df.columns)}")
print(f"\n✓ Dataset is CLEAN and ready for analysis!")


In [ ]:
print("Statistical Summary:")
print("=" * 50)
df.describe().round(2)

## 4. Understanding the Features

| Feature | Description | Values |
|---------|-------------|--------|
| age | Age in years | 29-76 |
| sex | Gender | 0=Female, 1=Male |
| cp | Chest Pain Type | 0=Typical Angina, 1=Atypical, 2=Non-anginal, 3=Asymptomatic |
| trestbps | Resting Blood Pressure (mm Hg) | 94-200 |
| chol | Serum Cholesterol (mg/dl) | 126-564 |
| fbs | Fasting Blood Sugar > 120 mg/dl | 0=No, 1=Yes |
| restecg | Resting ECG Results | 0=Normal, 1=ST-T abnormality, 2=LV hypertrophy |
| thalach | Maximum Heart Rate Achieved | 71-202 |
| exang | Exercise Induced Angina | 0=No, 1=Yes |
| oldpeak | ST Depression (exercise vs rest) | 0-6.2 |
| slope | Peak Exercise ST Slope | 0=Upsloping, 1=Flat, 2=Downsloping |
| ca | Major Vessels Colored by Fluoroscopy | 0-3 |
| thal | Thalassemia | 0=Normal, 1=Fixed Defect, 2=Reversible Defect |
| **target** | **Heart Disease** | **0=Healthy, 1=Disease** |

## 5. Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
target_counts = df['target'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Healthy (0)', 'Disease (1)'], target_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Target Distribution', fontsize=14)
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=['Healthy', 'Heart Disease'], 
            colors=colors, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12}, explode=[0, 0.05])
axes[1].set_title('Target Proportion', fontsize=14)

plt.tight_layout()
plt.show()

print(f"Class Balance: {target_counts[0]} Healthy vs {target_counts[1]} Disease")
print(f"Ratio: {target_counts[1]/target_counts[0]:.2f} — reasonably balanced!")

## 6. Univariate Analysis — Feature Distributions

In [ ]:
# Continuous features
continuous = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(continuous):
    axes[idx].hist(df[col], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean={df[col].mean():.1f}')
    axes[idx].set_title(f'{col} Distribution', fontsize=12)
    axes[idx].set_xlabel(col)
    axes[idx].legend()

axes[5].axis('off')
plt.suptitle('Distribution of Continuous Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features
categorical = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for idx, col in enumerate(categorical):
    counts = df[col].value_counts().sort_index()
    axes[idx].bar(counts.index, counts.values, color='coral', edgecolor='black')
    axes[idx].set_title(f'{col}', fontsize=12)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')

plt.suptitle('Distribution of Categorical Features', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Bivariate Analysis — Features vs Target

In [ ]:
# Age vs Target
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution by target
for t, color, label in [(0, '#2ecc71', 'Healthy'), (1, '#e74c3c', 'Disease')]:
    axes[0].hist(df[df['target']==t]['age'], bins=15, alpha=0.6, color=color, label=label, edgecolor='black')
axes[0].set_title('Age Distribution by Target', fontsize=12)
axes[0].set_xlabel('Age')
axes[0].legend()

# Max Heart Rate vs Target
for t, color, label in [(0, '#2ecc71', 'Healthy'), (1, '#e74c3c', 'Disease')]:
    axes[1].hist(df[df['target']==t]['thalach'], bins=15, alpha=0.6, color=color, label=label, edgecolor='black')
axes[1].set_title('Max Heart Rate by Target', fontsize=12)
axes[1].set_xlabel('thalach')
axes[1].legend()

# Oldpeak vs Target
for t, color, label in [(0, '#2ecc71', 'Healthy'), (1, '#e74c3c', 'Disease')]:
    axes[2].hist(df[df['target']==t]['oldpeak'], bins=15, alpha=0.6, color=color, label=label, edgecolor='black')
axes[2].set_title('ST Depression by Target', fontsize=12)
axes[2].set_xlabel('oldpeak')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Categorical features vs Target
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

cat_features = ['sex', 'cp', 'exang', 'slope', 'ca', 'thal']

for idx, col in enumerate(cat_features):
    ct = pd.crosstab(df[col], df['target'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[idx], color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[idx].set_title(f'{col} vs Target', fontsize=12)
    axes[idx].set_ylabel('Percentage')
    axes[idx].legend(['Healthy', 'Disease'])
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=0)

plt.suptitle('Categorical Features vs Heart Disease', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 9))
correlation = df.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

# Top correlations with target
print("\nCorrelation with Target (sorted by absolute value):")
print("=" * 50)
target_corr = correlation['target'].drop('target').abs().sort_values(ascending=False)
for feat, corr in target_corr.items():
    direction = correlation['target'][feat]
    print(f"  {feat:12s}: {direction:+.3f} {'↑ increases risk' if direction > 0 else '↓ decreases risk'}")

## 9. Outlier Detection (IQR Method)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
box_features = ['trestbps', 'chol', 'thalach', 'oldpeak']

for idx, col in enumerate(box_features):
    axes[idx].boxplot(df[col], vert=True)
    axes[idx].set_title(col)

plt.suptitle('Box Plots — Detecting Outliers', fontsize=13)
plt.tight_layout()
plt.show()

# Count outliers using IQR
print("\nOutlier Count (IQR Method):")
print("-" * 40)
for col in box_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"  {col:10s}: {outliers} outliers")

## 10. Data Preprocessing (Train-Test Split & Scaling)

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

In [ ]:
# Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")
print(f"\nTraining target distribution:")
print(f"  Healthy: {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)")
print(f"  Disease: {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)")
print(f"\nTest target distribution:")
print(f"  Healthy: {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%)")
print(f"  Disease: {(y_test==1).sum()} ({(y_test==1).mean()*100:.1f}%)")
print("\n✓ Stratified split preserves class proportions!")

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature Scaling (StandardScaler):")
print("=" * 50)
print(f"\nBefore scaling (first sample): {X_train.iloc[0].values[:5]}")
print(f"After scaling  (first sample): {X_train_scaled[0][:5].round(3)}")
print(f"\nScaled mean ≈ 0: {X_train_scaled.mean(axis=0).round(3)[:5]}")
print(f"Scaled std  ≈ 1: {X_train_scaled.std(axis=0).round(3)[:5]}")
print("\n✓ All features now on same scale — important for KNN & SVM!")

In [ ]:
# Save preprocessed data for Day 19
np.savez('preprocessed_data.npz',
         X_train=X_train.values, X_test=X_test.values,
         X_train_scaled=X_train_scaled, X_test_scaled=X_test_scaled,
         y_train=y_train.values, y_test=y_test.values,
         feature_names=X.columns.values)

## 11. Key Insights from EDA

### Data Cleaning Summary:
| Problem | Solution | Rows/Cols Affected |
|---------|----------|-------------------|
| Irrelevant column (`patient_id`) | Dropped | 1 column removed |
| Duplicate rows | `drop_duplicates()` | 8 rows removed |
| Missing `age` | Filled with **mean** | 8 values |
| Missing `chol`, `trestbps` | Filled with **median** | 18 values |
| Missing `thalach` | Filled with **group mean** | 10 values |
| Missing `oldpeak` | Dropped rows | 5 rows removed |

### Observations:
1. **Dataset is balanced** — ~46% Healthy, ~54% Disease → no need for SMOTE
2. **Top risk indicators** (highest correlation with target):
   - Exercise-induced angina (exang) ↑
   - Number of vessels (ca) ↑
   - Chest pain type (cp) ↑
   - ST depression (oldpeak) ↑
3. **Protective factors:**
   - Higher max heart rate (thalach) → less risk
4. **Outliers present** in cholesterol and blood pressure

### Next Steps (Day 19):
- Build 5 ML models (Logistic Regression, KNN, Decision Tree, Random Forest, SVM)
- Apply accuracy improvement techniques (GridSearchCV, Voting Classifier)
- Compare all models